<a href="https://colab.research.google.com/github/Elenadr/evaluaciondelmodelo/blob/main/ResultadosNFCySUB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import random
import pandas as pd
import numpy as np
from google.colab import drive
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader, Dataset
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
import uuid

# Montar Google Drive
drive.mount('/content/drive')

# Instalar dependencias
!pip install transformers datasets torch pandas numpy

# Definir clases
sub_classes = ["Frecuencia Insegura", "Sin Cifrado", "Modulación Insegura", "Código Fijo", "Segura"]
nfc_classes = ["Claves por defecto", "Lectura completa", "Escritura posible", "UID clonable", "No detectada"]

# Definir claves por defecto
default_keys = {
    "FF FF FF FF FF FF",
    "A0 A1 A2 A3 A4 A5",
    "D3 F7 D3 F7 D3 F7",
    "00 00 00 00 00 00",
    "B0 B1 B2 B3 B4 B5",
    "4D 3A 99 C3 51 DD"
}

# Funciones de etiquetado (sin cambios para SUB)
def label_sub_file(content, debug=False):
    lines = content.split('\n')
    vulnerabilities = []
    frecuencia_hz = None
    preset = ""
    protocolo = ""
    key = ""
    has_raw_data = any("RAW_Data" in line for line in lines)

    for line in lines:
        if line.startswith("Frequency:"):
            try:
                frecuencia_hz = int(line.split(":")[1].strip())
            except ValueError:
                frecuencia_hz = None
        elif line.startswith("Preset:"):
            preset = line.split(":")[1].strip()
        elif line.startswith("Protocol:"):
            protocolo = line.split(":")[1].strip()
        elif line.startswith("Key:"):
            key = line.split(":", 1)[1].strip()

    if debug:
        print(f"Debug: Frequency={frecuencia_hz}, Preset={preset}, Protocol={protocolo}, Key={key}, Has RAW={has_raw_data}")

    secure_frequencies = [868350000, 915000000]
    secure_presets = ["FuriHalSubGhzPresetAES"]
    secure_protocols = ["AES-Encrypted", "KeeLoq", "KeeLoq-Secure"]
    prob_rolling_code = 0.0
    prob_codigo_fijo = 0.0

    if frecuencia_hz in [433920000, 315000000]:
        vulnerabilities.append("Frecuencia Insegura")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Frecuencia Insegura")

    insecure_protocols = ["OOK", "RAW"]
    if any(p.lower() in protocolo.lower() for p in insecure_protocols) and protocolo not in secure_protocols:
        vulnerabilities.append("Sin Cifrado")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Sin Cifrado")

    insecure_presets = ["FuriHalSubGhzPresetOok650Async", "FuriHalSubGhzPresetOok270Async"]
    if preset in insecure_presets:
        vulnerabilities.append("Modulación Insegura")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Modulación Insegura")

    if has_raw_data:
        raw_lines = [line for line in lines if "RAW_Data" in line]
        raws = [re.findall(r"[-\d]+", line) for line in raw_lines]
        if len(raws) >= 2:
            repetition_count = sum(1 for i in range(len(raws)-1) if raws[i] == raws[i+1])
            if repetition_count >= len(raws) * 0.5:
                prob_codigo_fijo = max(prob_codigo_fijo, 1.0)
                if debug:
                    print("Debug: High repetition in RAW_Data, setting prob_codigo_fijo=1.0")
            elif len(raw_lines) > 10 and repetition_count == 0:
                prob_rolling_code = 0.7
                if debug:
                    print("Debug: Long non-repeating RAW_Data, setting prob_rolling_code=0.7")
    else:
        if protocolo in ["KeeLoq", "KeeLoq-Secure"]:
            prob_rolling_code = 0.9
            if debug:
                print("Debug: KeeLoq protocol, setting prob_rolling_code=0.9")
        else:
            key_parts = key.split()
            if len(key_parts) > 4 and len(set(key_parts)) < len(key_parts) / 2:
                prob_codigo_fijo = max(prob_codigo_fijo, 0.8)
                if debug:
                    print("Debug: Repetitive key pattern, setting prob_codigo_fijo=0.8")

    if prob_codigo_fijo >= 0.8:
        vulnerabilities.append("Código Fijo")
        if debug:
            print("Debug: Added Código Fijo")

    if (not vulnerabilities and
        frecuencia_hz in secure_frequencies and
        preset in secure_presets and
        protocolo in secure_protocols and
        not has_raw_data):
        vulnerabilities = ["Segura"]
        if debug:
            print("Debug: No vulnerabilities and secure configuration, labeling as Segura")
    else:
        if prob_rolling_code >= 0.8 and debug:
            print("Debug: High prob_rolling_code but vulnerabilities present, keeping vulnerabilities")
        prob_rolling_code = min(prob_rolling_code, 0.2) if vulnerabilities else prob_rolling_code
        if debug:
            print("Debug: Vulnerabilities detected, adjusting prob_rolling_code")

    if debug:
        print(f"Debug: Final vulnerabilities={vulnerabilities}, Prob Rolling Code={prob_rolling_code}")

    return vulnerabilities, prob_rolling_code

# Función corregida para NFC
def label_nfc_file(content, debug=False):
    lines = content.split('\n')
    vulnerable_uid = False
    bloques_leidos = 0
    claves_encontradas = set()
    tipo = None
    pages_read = 0
    pages_total = 0
    has_password = False
    counter_0 = 0
    counter_1 = 0
    counter_2 = 0

    for line in lines:
        if line.startswith("Device type:") or line.startswith("Type:"):
            tipo = line.strip().split(":", 1)[1].strip()
        elif re.search(r'Block \d+:', line) and "MIFARE Classic" in tipo:
            bloques_leidos += 1
            data = line.split(':', 1)[1].strip().upper()
            if any(key in data for key in default_keys):
                claves_encontradas.add(data)
        elif re.search(r'Key [AB]:', line) and "MIFARE Classic" in tipo:
            key = line.split(':', 1)[1].strip().upper()
            if any(key in dk for dk in default_keys):
                claves_encontradas.add(key)
        elif line.startswith("Pages total:"):
            pages_total = int(line.split(":")[1].strip())
        elif line.startswith("Pages read:"):
            pages_read = int(line.split(":")[1].strip())
        elif "Password:" in line or "PACK:" in line:
            has_password = True
        elif line.startswith("Counter 0:"):
            counter_0 = int(line.split(":")[1].strip())
        elif line.startswith("Counter 1:"):
            counter_1 = int(line.split(":")[1].strip())
        elif line.startswith("Counter 2:"):
            counter_2 = int(line.split(":")[1].strip())

    if debug:
        print(f"Debug: Type={tipo}, Bloques leídos={bloques_leidos}, Claves encontradas={claves_encontradas}, Pages read={pages_read}/{pages_total}, Has password={has_password}, Counters={counter_0},{counter_1},{counter_2}")

    vulnerabilidades = []
    if "MIFARE Classic" in tipo:
        vulnerable_uid = True
        if claves_encontradas:
            vulnerabilidades.append("Claves por defecto")
        if bloques_leidos >= 8:
            vulnerabilidades.append("Lectura completa")
        if claves_encontradas and bloques_leidos > 0:
            vulnerabilidades.append("Escritura posible")
    elif "NTAG" in tipo:
        if pages_read >= pages_total and pages_total > 0:
            vulnerabilidades.append("Lectura completa")
            # Solo asignar "Escritura posible" si no hay contraseña y hay evidencia de escritura reciente
            if not has_password and (counter_0 > 0 or counter_1 > 0 or counter_2 > 0):
                vulnerabilidades.append("Escritura posible")
        vulnerable_uid = False
    else:
        vulnerable_uid = False

    if vulnerable_uid:
        vulnerabilidades.append("UID clonable")

    if not vulnerabilidades:
        vulnerabilidades = ["No detectada"]

    if debug:
        print(f"Debug: Vulnerabilidades={vulnerabilidades}")

    return vulnerabilidades

# Cargar modelos y tokenizer
tokenizer = BertTokenizer.from_pretrained('/content/drive/MyDrive/models/tokenizer')
sub_model = BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/models/sub_model')
nfc_model = BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/models/nfc_model')

# Función de predicción (sin cambios)
def predict_vulnerabilities(file_path, threshold=0.3):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
    except UnicodeDecodeError:
        with open(file_path, 'r', encoding='latin-1') as f:
            content = f.read().strip()
    except Exception as e:
        print(f"Error al leer {file_path}: {str(e)}")
        return ["Error al procesar el archivo"]

    model = sub_model if file_path.endswith('.sub') else nfc_model
    classes = sub_classes if file_path.endswith('.sub') else nfc_classes

    # Extraer atributos
    frecuencia_hz = None
    preset = ""
    protocolo = ""
    tipo = ""
    has_raw_data = "RAW_Data" in content
    has_password = "Password:" in content or "PACK:" in content
    bloques_leidos = len([line for line in content.split('\n') if re.search(r'Block \d+:', line)])
    claves_encontradas = any(key in content.upper() for key in default_keys)
    pages_read = 0  # Añadido: Inicializar pages_read
    pages_total = 0  # Añadido: Inicializar pages_total

    for line in content.split('\n'):
        if line.startswith("Frequency:"):
            try:
                frecuencia_hz = int(line.split(":")[1].strip())
            except ValueError:
                frecuencia_hz = None
        elif line.startswith("Preset:"):
            preset = line.split(":")[1].strip()
        elif line.startswith("Protocol:"):
            protocolo = line.split(":")[1].strip()
        elif line.startswith("Device type:") or line.startswith("Type:"):
            tipo = line.strip().split(":", 1)[1].strip()

    extra_features = f"Bloques leidos: {bloques_leidos} Claves encontradas: {claves_encontradas} Tipo: {tipo}"
    content_with_features = content + "\n" + extra_features

    encodings = tokenizer(content_with_features, padding=True, truncation=True, max_length=512, return_tensors='pt')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    encodings = {key: val.to(device) for key, val in encodings.items()}
    model.to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        # Usar sigmoid para ambos casos (NFC y SUB) para permitir clasificación multi-etiqueta
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    print(f"\nProbabilidades para {file_path}:")
    for cls, prob in zip(classes, probs):
        print(f"{cls}: {prob:.4f}")

    predicted_vulns = [classes[i] for i, prob in enumerate(probs) if prob > threshold]

    # Reglas basadas en características extraídas para NFC
    if file_path.endswith('.nfc'):
      if claves_encontradas and "Claves por defecto" not in predicted_vulns:
          predicted_vulns.append("Claves por defecto")
      if bloques_leidos >= 8 and "Lectura completa" not in predicted_vulns:
          predicted_vulns.append("Lectura completa")
      if "MIFARE Classic" in tipo and "UID clonable" not in predicted_vulns:
          predicted_vulns.append("UID clonable")
      # Nueva regla: Evitar "UID clonable" para MIFARE DESFire
      if "MIFARE DESFire" in tipo and "UID clonable" in predicted_vulns:
          predicted_vulns.remove("UID clonable")
      # Nueva regla: Evitar "Lectura completa" y "Escritura posible" si no hay bloques/páginas leídas
      if bloques_leidos == 0 and pages_read == 0:
          if "Lectura completa" in predicted_vulns:
              predicted_vulns.remove("Lectura completa")
          if "Escritura posible" in predicted_vulns:
              predicted_vulns.remove("Escritura posible")

    final_vulns = predicted_vulns.copy()
    if file_path.endswith('.sub'):
        if any(v in final_vulns for v in ["Frecuencia Insegura", "Sin Cifrado", "Modulación Insegura", "Código Fijo"]):
            if "Segura" in final_vulns:
                final_vulns.remove("Segura")
    else:
        if any(v in final_vulns for v in ["Claves por defecto", "Lectura completa", "Escritura posible", "UID clonable"]):
            if "No detectada" in final_vulns:
                final_vulns.remove("No detectada")

    if not final_vulns:
        final_vulns = ["Ninguna vulnerabilidad detectada"]

    print("\n" + "="*50)
    print(f"📄 Análisis de: {file_path}")
    print("="*50)
    print("\n🚨 Vulnerabilidades detectadas:")
    print("-"*40)
    for vuln in final_vulns:
        print(f"• {vuln}")
    if file_path.endswith('.sub'):
        _, prob_rolling_code = label_sub_file(content, debug=True)
        print(f"• Probabilidad de Código Variable: {prob_rolling_code:.2f}")
    print("-"*40)

    if final_vulns != ["Ninguna vulnerabilidad detectada"]:
        print("\n💥 Métodos de explotación sugeridos:")
        print("-"*40)
        if file_path.endswith('.sub'):
            for vuln in final_vulns:
                if vuln == "Frecuencia Insegura":
                    print(f"🔧 {vuln}:")
                    print(f"   Frecuencia: {frecuencia_hz} Hz")
                    print("   Explotación: Capturar y retransmitir la señal para un ataque de repetición.")
                    print("   Herramientas: Flipper Zero, HackRF One, Yard Stick One")
                elif vuln == "Sin Cifrado":
                    print(f"🔧 {vuln}:")
                    print(f"   Protocolo: {protocolo}")
                    print("   Explotación: Interceptar y decodificar datos en texto plano.")
                    print("   Herramientas: URH, Flipper Zero, RTL-SDR")
                elif vuln == "Modulación Insegura":
                    print(f"🔧 {vuln}:")
                    print(f"   Preset: {preset}")
                    print("   Explotación: Jamming o suplantación de señal debido a modulación predecible.")
                    print("   Herramientas: Flipper Zero, HackRF One")
                elif vuln == "Código Fijo":
                    print(f"🔧 {vuln}:")
                    print("   Explotación: Clonar código fijo capturando y retransmitiéndolo.")
                    print("   Herramientas: Flipper Zero, Proxmark3, SDR")
                elif vuln == "Segura":
                    print(f"🔧 {vuln}:")
                    print("   Nota: Protocolo seguro detectado. Verificar implementación del receptor.")
        else:
            for vuln in final_vulns:
                if vuln == "Claves por defecto":
                    print(f"🔧 {vuln}:")
                    print(f"   Tipo: {tipo}")
                    print("   Explotación: Usar claves por defecto para acceder a datos o modificar tarjeta.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "Lectura completa":
                    print(f"🔧 {vuln}:")
                    print(f"   Bloques leídos: {bloques_leidos}")
                    print("   Explotación: Extraer todos los datos de la tarjeta para análisis o clonación.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "Escritura posible":
                    print(f"🔧 {vuln}:")
                    print("   Explotación: Modificar datos de la tarjeta para alterar su comportamiento.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "UID clonable":
                    print(f"🔧 {vuln}:")
                    print(f"   Tipo: {tipo}")
                    print("   Explotación: Clonar UID para replicar tarjeta.")
                    print("   Herramientas: Proxmark3, Flipper Zero")
                elif vuln == "No detectada":
                    print(f"🔧 {vuln}:")
                    print("   Nota: No se detectaron vulnerabilidades. Verificar configuración física.")
        print("-"*40)

    print("\n" + "="*50 + "\n")
    return final_vulns

# Testear archivo problemático
test_file = "/content/drive/MyDrive/EvalModelNFC/eval_nfc_todas_vuln_4.nfc"
predict_vulnerabilities(test_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl (211.5 MB)

Probabilidades para /content/drive/MyDrive/EvalModelNFC/eval_nfc_todas_vuln_4.nfc:
Claves por defecto: 0.2608
Lectura completa: 0.6434
Escritura posible: 0.4291
UID clonable: 0.5013
No detectada: 0.1511

📄 Análisis de: /content/drive/MyDrive/EvalModelNFC/eval_nfc_todas_vuln_4.nfc

🚨 Vulnerabilidades detectadas:
----------------------------------------
• Lectura completa
• Escritura posible
• UID clonable
• Claves por defecto
----------------------------------------

💥 Métodos de explotación sugeridos:
----------------------------------------
🔧 Lectura completa:
   Bloques leídos: 16
   Explotación: Extraer todos los datos de la tarjeta para análisis o clonación.
   Herramientas: Pro

['Lectura completa', 'Escritura posible', 'UID clonable', 'Claves por defecto']

In [29]:
# Testear archivo problemático
test_file = "/content/drive/MyDrive/EvalModelSUB/CAME-12bit-433-13.sub"
predict_vulnerabilities(test_file)


Probabilidades para /content/drive/MyDrive/EvalModelSUB/CAME-12bit-433-13.sub:
Frecuencia Insegura: 0.7570
Sin Cifrado: 0.9978
Modulación Insegura: 0.9967
Código Fijo: 0.8272
Segura: 0.0029

📄 Análisis de: /content/drive/MyDrive/EvalModelSUB/CAME-12bit-433-13.sub

🚨 Vulnerabilidades detectadas:
----------------------------------------
• Frecuencia Insegura
• Sin Cifrado
• Modulación Insegura
• Código Fijo
Debug: Frequency=433920000, Preset=FuriHalSubGhzPresetOok650Async, Protocol=RAW, Key=, Has RAW=True
Debug: Added Frecuencia Insegura
Debug: Added Sin Cifrado
Debug: Added Modulación Insegura
Debug: Long non-repeating RAW_Data, setting prob_rolling_code=0.7
Debug: Added Código Fijo
Debug: Vulnerabilities detected, adjusting prob_rolling_code
Debug: Final vulnerabilities=['Frecuencia Insegura', 'Sin Cifrado', 'Modulación Insegura', 'Código Fijo'], Prob Rolling Code=0.2
• Probabilidad de Código Variable: 0.20
----------------------------------------

💥 Métodos de explotación sugeridos:


['Frecuencia Insegura', 'Sin Cifrado', 'Modulación Insegura', 'Código Fijo']